In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Matrix Completion for Movie Recommendations

## What Are We Doing?

**Goal:** Build a recommendation system using **matrix completion**.

Think of it like this:
- You have 943 users and 1,682 movies
- Only ~100,000 ratings exist (6% filled)
- Question: **What would users rate movies they haven't seen?**
- Solution: Use math to *guess* the missing ratings

This notebook walks through the entire process step-by-step.

## Step 1: Load & Prepare Data

We need:
- **u.data**: User ratings (user_id, movie_id, rating, timestamp)
- **u.items**: Movie metadata (title, year, genres)

Then we'll create a **user-item matrix** where:
- Rows = users
- Columns = movies
- Values = ratings (1-5) or NaN (not rated)

In [5]:
import pandas as pd
import numpy as np

# Load u.data (user ratings)
u_data_cols = ['user_id', 'item_id', 'rating', 'timestamp']
u_data_df = pd.read_csv('/content/drive/My Drive/Colab Notebooks/AI/u.data', sep='\t', names=u_data_cols)

# Load u.items (movie metadata)
u_items_cols = ['item_id', 'movie_title', 'release_date', 'video_release_date',
                'IMDb_URL', 'unknown', 'Action', 'Adventure', 'Animation',
                'Childrens', 'Comedy', 'Crime', 'Documentary', 'Drama',
                'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery',
                'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
u_items_df = pd.read_csv('/content/drive/My Drive/Colab Notebooks/AI/u.item', sep='|', names=u_items_cols, encoding='latin-1')

print(f'✓ Loaded {len(u_data_df)} ratings')
print(f'✓ Loaded {len(u_items_df)} movies')

✓ Loaded 100000 ratings
✓ Loaded 1682 movies


In [6]:
# Create user-item matrix: rows=users, columns=movies, values=ratings
user_item_matrix = u_data_df.pivot_table(index='user_id', columns='item_id', values='rating')

print(f'Matrix shape: {user_item_matrix.shape}')
print(f'  - {user_item_matrix.shape[0]} users')
print(f'  - {user_item_matrix.shape[1]} movies')
print(f'  - {user_item_matrix.count().sum()} ratings')

# Show a small sample
display(user_item_matrix.head())

Matrix shape: (943, 1682)
  - 943 users
  - 1682 movies
  - 100000 ratings


item_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 2: Understand Sparsity

Our matrix is mostly **empty (NaN)**. This is normal for recommendation systems.

**Sparsity** = percentage of missing values
- 90%+ sparse = very sparse (typical)
- Why? Because most users haven't rated most movies

In [7]:
num_users, num_movies = user_item_matrix.shape
num_ratings = user_item_matrix.count().sum()
total_possible = num_users * num_movies

sparsity = (1 - num_ratings / total_possible) * 100

print(f'Sparsity: {sparsity:.1f}%')
print(f'  Filled cells: {num_ratings:,}')
print(f'  Total cells: {total_possible:,}')
print(f'\nInterpretation: {100-sparsity:.1f}% of the matrix has ratings')

Sparsity: 93.7%
  Filled cells: 100,000
  Total cells: 1,586,126

Interpretation: 6.3% of the matrix has ratings


## Step 3: Normalize Ratings (Mean Centering)

**Why?** Different users have different rating scales:
- Bob always gives 4-5 stars (generous)
- Alice always gives 1-3 stars (harsh)

**Solution:** Subtract each user's average from their ratings
- Result: Ratings centered around 0
- Positive = better than their average
- Negative = worse than their average

In [8]:
# Calculate mean rating per user
user_means = user_item_matrix.mean(axis=1)
print(f'Average rating per user (first 5):')
print(user_means.head())

# Subtract mean from each rating
normalized_matrix = user_item_matrix.sub(user_means, axis=0)

print(f'\nNormalized matrix (first 5 users):')
display(normalized_matrix.head())

Average rating per user (first 5):
user_id
1    3.610294
2    3.709677
3    2.796296
4    4.333333
5    2.874286
dtype: float64

Normalized matrix (first 5 users):


item_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.389706,-0.610294,0.389706,-0.610294,-0.610294,1.389706,0.389706,-2.610294,1.389706,-0.610294,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.290323,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.709677,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1.125714,0.125714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 4: Fill Missing Values (Matrix Completion)

**Algorithm: SoftImpute**

How it works:
1. Start with sparse matrix (lots of NaN)
2. Iteratively fill NaN values based on patterns in rated movies
3. Find low-rank representation (compress information)
4. Repeat until convergence

Result: A **dense matrix** with predictions for all NaN values

In [9]:
!pip install -q fancyimpute

from fancyimpute import SoftImpute

print('Running SoftImpute (this may take 30-60 seconds)...')
imputer = SoftImpute(verbose=True, max_iters=100)
imputed_values = imputer.fit_transform(normalized_matrix.values)

# Convert back to DataFrame
imputed_matrix = pd.DataFrame(imputed_values,
                              index=normalized_matrix.index,
                              columns=normalized_matrix.columns)

print('\n✓ Matrix completion done!')

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.5 MB/s eta 0:00:00
Running SoftImpute (this may take 30-60 seconds)...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[SoftImpute] Max Singular Value of X_init = 81.836389
[SoftImpute] Iter 1: observed MAE=0.094216 rank=756
[SoftImpute] Iter 2: observed MAE=0.092661 rank=671
[SoftImpute] Iter 3: observed MAE=0.091269 rank=616
[SoftImpute] Iter 4: observed MAE=0.090118 rank=576
[SoftImpute] Iter 5: observed MAE=0.089142 rank=545
[SoftImpute] Iter 6: observed MAE=0.088281 rank=520
[SoftImpute] Iter 7: observed MAE=0.087538 rank=498
[SoftImpute] Iter 8: observed MAE=0.086892 rank=480
[SoftImpute] Iter 9: observed MAE=0.086324 rank=464
[SoftImpute] Iter 10: observed MAE=0.085805 rank=451
[SoftImpute] Iter 11: observed MAE=0.085327 rank=437
[SoftImpute] Iter 12: observed MAE=0.084892 rank=426
[SoftImpute] Iter 13: observed MAE=0.084512 rank=415
[SoftImpute] Iter 14: observed MAE=0.084185 rank=407
[SoftImpute] Iter 15: observed MAE=0.083890 rank=399
[SoftImpute] Iter 16: observed MAE=0.083596 rank=390
[SoftImpute] Iter 17: observed MAE=0.083323 rank=383
[SoftImpute] Iter 18: observed MAE=0.083073 rank=377
[

## Step 5: Denormalize (Add Back User Means)

We normalized by subtracting user means. Now reverse it:

**Final predictions = imputed_values + user_mean**

Result: Predicted ratings in the 1-5 star scale

In [10]:
# Add back user means
final_predictions = imputed_matrix.add(user_means, axis=0)

# Clip to valid rating range [1, 5]
final_predictions = final_predictions.clip(1, 5)

print('Final predicted ratings (1-5 scale):')
display(final_predictions.head())

Final predicted ratings (1-5 scale):


item_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.000000,3.000000,4.000000,3.000000,3.000000,5.000000,4.000000,1.000000,5.000000,3.000000,...,3.601282,3.611066,3.607432,3.601200,3.610653,3.483243,3.603018,3.543131,3.610661,3.576758
2,4.000000,3.449808,3.471684,3.491167,3.733502,3.967496,3.530777,3.918459,3.877957,2.000000,...,3.684339,3.715506,3.697913,3.672302,3.707580,3.690618,3.708586,3.699602,3.710293,3.707860
3,2.713276,2.794470,2.796984,2.369883,2.606293,2.923576,2.974133,3.045511,3.206870,2.385144,...,2.828520,2.795214,2.759342,2.678889,2.795854,2.779387,2.795328,2.787358,2.796074,2.786412
4,4.412968,4.326302,4.378921,4.016096,4.422969,4.355896,4.455173,4.555087,3.813739,4.504445,...,4.415694,4.329199,4.335537,4.340333,4.324870,4.345820,4.334048,4.339934,4.334171,4.328061
5,4.000000,3.000000,2.373838,3.324905,2.860753,2.754117,4.147345,3.111374,3.032924,2.625854,...,3.019070,2.871009,2.856724,2.818489,2.861545,2.821286,2.871251,2.846268,2.874968,2.897862


## Step 6: Generate Recommendations

Now we can recommend movies to any user:
1. Pick a user
2. Get movies they haven't rated
3. Sort by predicted rating (highest first)
4. Return top 5 as recommendations

In [11]:
def get_recommendations(user_id, n_recommendations=5):
    # Movies this user hasn't rated
    unseen_movies = user_item_matrix.loc[user_id][user_item_matrix.loc[user_id].isna()].index

    if len(unseen_movies) == 0:
        print(f'User {user_id} has rated all movies!')
        return

    # Get predictions for unseen movies
    predictions = final_predictions.loc[user_id, unseen_movies]

    # Sort and get top N
    top_predictions = predictions.nlargest(n_recommendations)

    # Get movie titles
    recommendations = []
    for movie_id, pred_rating in top_predictions.items():
        movie_title = u_items_df[u_items_df['item_id'] == movie_id]['movie_title'].values[0]
        recommendations.append((movie_title, pred_rating))

    return recommendations

# Example: Get recommendations for user 1
user_id = 1
recs = get_recommendations(user_id)

print(f'\n🎬 Top recommendations for User {user_id}:\n')
for i, (title, rating) in enumerate(recs, 1):
    print(f'{i}. {title} (Predicted: {rating:.1f}⭐)')


🎬 Top recommendations for User 1:

1. Secrets & Lies (1996) (Predicted: 5.0⭐)
2. L.A. Confidential (1997) (Predicted: 5.0⭐)
3. Leaving Las Vegas (1995) (Predicted: 4.9⭐)
4. Close Shave, A (1995) (Predicted: 4.9⭐)
5. Rear Window (1954) (Predicted: 4.9⭐)


## Summary: What We Did

1. **Loaded data** → 943 users × 1,682 movies
2. **Created matrix** → Sparse (93% empty)
3. **Normalized** → Center around user mean
4. **Completed matrix** → SoftImpute filled gaps
5. **Denormalized** → Back to 1-5 scale
6. **Generated recommendations** → Top predicted movies

### Key Ideas

- **Matrix completion** = predicting missing values based on patterns
- **Normalization** = handle different rating scales
- **Low-rank approximation** = compress noisy data
- Result: Fast, interpretable recommendations